## POT NTR, Time Varying

In this section, 
1. first the POT of NTR time series and the associated thresholds of POT for each year are achieved. 
3. Based on POT NTR times, a 2.5 days before and after each time is considered to find the max of RF (and the accumualted values) and RD values.

* RF and NTR are hourly and RD is daily

In [1]:
import numpy as np
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from datetime import timedelta
from pathlib import Path

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_ntr = 'time'
time_col_rf = 'time'
time_col_rd = 'dateTime'

# Name of variable column in each dataset
ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
target_events = 5  # per-year target

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:

    base_path = core_directory / Path(f'{loc}')
    # Load & basic time fields
    NTR_File = rf'{dataset_directory}/In_Situ/NOAA_Tide_Gauge_MSL_All_Predicted.csv'
    RF_File = rf'{dataset_directory}/In_Situ/GHCN_Rainfall_1950_2050_hourly_accumulated.csv'
    RD_File = rf'{dataset_directory}/In_Situ/usgs_discharge.csv'

    df_ntr = pd.read_csv(NTR_File)
    df_ntr['time'] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr['Year'] = df_ntr['time'].dt.year

    #Rainfall
    df_rf = pd.read_csv(RF_File)
    df_rf['time'] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index('time')

    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    ntr_series = df_ntr[ntr_col].dropna()
    decluster_window = timedelta(days=2.5)
    output_dir = rf'{base_path}/TimeVarying'
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Storage
    # ────────────────────────────────────────────────────────────────────────────────
    all_pot = []
    thresholds = []
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Yearly POT extraction + NTR matching
    # ────────────────────────────────────────────────────────────────────────────────
    for year, group in ntr_series.groupby(ntr_series.index.year):
        group_sorted = group.sort_values(ascending=False)
        selected = []
    
        for idx, val in group_sorted.items():
            if any(abs((idx - sel).days) < decluster_window.days for sel in selected):
                continue
            selected.append(idx)
            if len(selected) == target_events:
                break
    
        if len(selected) < target_events:
            continue
    
        # Minimum of selected events becomes the threshold for that year
        pot_values = [ntr_series.loc[t] for t in selected]
        threshold_value = min(pot_values)
        thresholds.append({'Year': year, 'Threshold': threshold_value})
    
        # Match with NTR
        for t, val in zip(selected, pot_values):
            start = t - decluster_window
            end = t + decluster_window
            rf_window = df_rf.loc[start:end]
            max_rf = rf_window[acc_hr].max() if not rf_window.empty else np.nan
            all_pot.append({'time': t, 'POT_NTR': val, 'Max_RF_in_5d': max_rf, 'Year': year})
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Save outputs
    # ────────────────────────────────────────────────────────────────────────────────
    df_pot = pd.DataFrame(all_pot).sort_values('time').reset_index(drop=True)
    df_thresh = pd.DataFrame(thresholds)
    
    df_pot.to_csv(os.path.join(output_dir, f'POT_NTR_TimeVarying_{target_events}.csv'), index=False)
    df_thresh.to_csv(os.path.join(output_dir, f'NTR_TimeVarying_Thresholds_{target_events}.csv'), index=False)
    
    
    #River Discharge
    POT_df = pd.read_csv(rf'{output_dir}/POT_NTR_TimeVarying_{target_events}.csv')
    POT_df['time'] = pd.to_datetime(POT_df['time'])
    POT_df = POT_df.set_index(['time'])
    
    df_RD = pd.read_csv(RD_File)
    df_RD['time'] = pd.to_datetime(df_RD[time_col_rd])
    df_RD = df_RD.drop(columns=[time_col_rd])
    df_RD = df_RD.set_index(['time'])
    
    # Ensure RD is daily (rounded to date)
    df_RD.index = df_RD.index.date  # Convert to datetime.date type
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in POT_df.index:
        event_date = event_time.date()  # extract the date part (YYYY-MM-DD)
    
        start_date = event_date - timedelta(days=2.5)
        end_date = event_date + timedelta(days=2.5)
    
        # Subset discharge data in that window
        window_data = df_RD.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    POT_df.index = pd.to_datetime(POT_df.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    POT_df = POT_df.copy()
    POT_df['date'] = POT_df.index.date  # datetime.date
    POT_df = POT_df.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  # 'date' is of type datetime.date
    
    # Step 3: Merge on 'date'
    merged_df = POT_df.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    merged_df = merged_df.drop(columns=['time'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['Time'] = pd.to_datetime(merged_df['Time'])
    
    merged_df.to_csv(rf'{output_dir}/POT_NTR_TimeVarying_RF_RD_{target_events}.csv', index=False)
    

C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\333386472.py:196: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\333386472.py:196: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')


## POT NTR, Stationary

In this section, 
1. first the POT of NTR time series based on a constant threshold is achieved. 
2. Based on POT NTR times, a 2.5 days before and after each time is considered to find the max of RF (and the accumualted values) and RD values.

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_ntr = 'time'
time_col_rf = 'time'
time_col_rd = 'dateTime'

# Name of variable column in each dataset
ntr_col = 'Storm_Surge_predicted_RF'
rd_col = 'discharge_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
target_events = 5  # per-year target

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:
    base_path = core_directory / Path(f'{loc}')
    
    NTR_File = rf'{dataset_directory}/In_Situ/NOAA_Tide_Gauge_MSL_All_Predicted.csv'
    RF_File = rf'{dataset_directory}/In_Situ/GHCN_Rainfall_1950_2050_hourly_accumulated.csv'
    RD_File = rf'{dataset_directory}/In_Situ/usgs_discharge.csv'

    df_ntr = pd.read_csv(NTR_File)
    df_ntr['time'] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index('time')

    df_rf = pd.read_csv(RF_File)
    df_rf['time'] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index('time')

    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    ntr_series = df_ntr[ntr_col].dropna()
    decluster_window = timedelta(days=2.5)
    output_dir = rf'{base_path}/Stationary'

    # ────────────────────────────────────────────────────────────────────────────────
    # Find constant threshold for 5 events/year
    # ────────────────────────────────────────────────────────────────────────────────
    n_years = ntr_series.index.year.nunique()
    total_target_events = target_events * n_years
    
    def count_events(threshold):
        selected = []
        for t, val in ntr_series.items():
            if val >= threshold:
                if any(abs((t - sel).days) < decluster_window.days for sel in selected):
                    continue
                selected.append(t)
        return len(selected)
    
    # Start from the initial threshold (top N values)
    sorted_values = ntr_series.sort_values(ascending=False)
    low, high = sorted_values.iloc[-1], sorted_values.iloc[0]  
    best_threshold = None
    
    # Binary search for the threshold that gives ~ target_events
    for _ in range(50):  # 50 iterations for convergence
        mid = (low + high) / 2
        events = count_events(mid)
        if events > total_target_events:
            low = mid
        else:
            high = mid
        best_threshold = mid
    
    threshold_value = best_threshold
    
    # Declustering window: ±2.5 days
    decluster_window = timedelta(days=2.5)
    
    # Step 1: Identify POT events (≥ threshold_value) with 2.5-day declustering
    pot_selected = []
    for t, val in ntr_series.sort_values(ascending=False).items():
        if val >= threshold_value:
            if any(abs((t - sel).days) <= decluster_window.days for sel in pot_selected):
                continue
            pot_selected.append(t)
    
    # Step 2: Mark all times within ±2.5 days of any POT event
    pot_exclusion_periods = [(event_time - decluster_window, event_time + decluster_window)
                             for event_time in pot_selected]
    
    def is_in_exclusion_period(time, exclusion_periods):
        return any(start <= time <= end for start, end in exclusion_periods)
    
    # Step 3: Filter below threshold and outside exclusion windows
    below_threshold_times = df_ntr[
        (df_ntr[ntr_col] < threshold_value) &
        (~df_ntr.index.map(lambda t: is_in_exclusion_period(t, pot_exclusion_periods)))
    ].index
    
    # Step 4: For each POT event, find max NTR in ±2.5 days
    pot_list = []
    for t in pot_selected:
        rf_window = df_rf.loc[t - decluster_window : t + decluster_window]
        max_rf= rf_window[acc_hr].max() if not rf_window.empty else np.nan
        pot_list.append({
            'time': t,
            'POT_NTR': df_ntr.loc[t, ntr_col],
            'Max_RF_in_5d': max_rf
        })
    
    df_pot_events = pd.DataFrame(pot_list).sort_values('time').reset_index(drop=True)
    
    output_below_file = os.path.join(output_dir, f'NTR_Stationary_Threshold_{round(threshold_value,2)}_{target_events}.csv')
    df_pot_events.to_csv(output_below_file, index=False)
    
    output_below_file = os.path.join(output_dir, f'POT_NTR_Stationary_{target_events}.csv')
    df_pot_events.to_csv(output_below_file, index=False)

    POT_df = pd.read_csv(rf'{output_dir}/POT_NTR_Stationary_{target_events}.csv')
    POT_df['time'] = pd.to_datetime(POT_df['time'])
    POT_df = POT_df.set_index(['time'])
    
    df_RD = pd.read_csv(RD_File)    
    df_RD['time'] = pd.to_datetime(df_RD[time_col_rd])
    df_RD = df_RD.drop(columns=[time_col_rd])
    df_RD = df_RD.set_index(['time'])
    
    # Ensure RD is daily (rounded to date)
    df_RD.index = df_RD.index.date 
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in POT_df.index:
        event_date = event_time.date() 
    
        start_date = event_date - timedelta(days=2.5)
        end_date = event_date + timedelta(days=2.5)
    
        # Subset discharge data in that window
        window_data = df_RD.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    POT_df.index = pd.to_datetime(POT_df.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    POT_df = POT_df.copy()
    POT_df['date'] = POT_df.index.date  
    POT_df = POT_df.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'}) 
    
    # Step 3: Merge on 'date'
    merged_df = POT_df.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    merged_df = merged_df.drop(columns=['time'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['Time'] = pd.to_datetime(merged_df['Time'])
    
    merged_df.to_csv(rf'{output_dir}/POT_NTR_Stationary_RF_RD_{target_events}.csv', index=False)
    

C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\1030619739.py:191: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\1030619739.py:191: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')


## POT NTR, Reanalysis, Time Varying

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_ntr = 'time'
time_col_rf = 'time'
time_col_rd = 'valid_time'

# Name of variable column in each dataset
ntr_col = 'Average'
rd_col = 'basin_avg_dis24_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
target_events = 10  

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:

    base_path = core_directory / Path(f'{loc}')
    # Load & basic time fields
    NTR_File = rf'{dataset_directory}/Reanalysis/Compare_Points_{loc}_Avg.csv'
    RF_File = rf'{dataset_directory}/Reanalysis/APCP_Basin_Average_{loc}_Combined_accumulated.csv'
    RD_File = rf'{dataset_directory}/Reanalysis/GloFAS_{loc}_Basin_Avg.csv'
    
    df_ntr = pd.read_csv(NTR_File)
    df_ntr['time'] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr['Year'] = df_ntr['time'].dt.year

    #Rainfall
    df_rf = pd.read_csv(RF_File)
    df_rf['time'] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index('time')

    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    ntr_series = df_ntr[ntr_col].dropna()
    decluster_window = timedelta(days=2.5)
    output_dir = rf'{base_path}/Reanalysis/TimeVarying'
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Storage
    # ────────────────────────────────────────────────────────────────────────────────
    all_pot = []
    thresholds = []
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Yearly POT extraction + NTR matching
    # ────────────────────────────────────────────────────────────────────────────────
    for year, group in ntr_series.groupby(ntr_series.index.year):
        group_sorted = group.sort_values(ascending=False)
        selected = []
    
        for idx, val in group_sorted.items():
            if any(abs((idx - sel).days) < decluster_window.days for sel in selected):
                continue
            selected.append(idx)
            if len(selected) == target_events:
                break
    
        if len(selected) < target_events:
            continue
    
        # Minimum of selected events becomes the threshold for that year
        pot_values = [ntr_series.loc[t] for t in selected]
        threshold_value = min(pot_values)
        thresholds.append({'Year': year, 'Threshold': threshold_value})
    
        # Match with NTR
        for t, val in zip(selected, pot_values):
            start = t - decluster_window
            end = t + decluster_window
            rf_window = df_rf.loc[start:end]
            max_rf = rf_window[acc_hr].max() if not rf_window.empty else np.nan
            all_pot.append({'time': t, 'POT_NTR': val, 'Max_RF_in_5d': max_rf, 'Year': year})
    
    # ────────────────────────────────────────────────────────────────────────────────
    # Save outputs
    # ────────────────────────────────────────────────────────────────────────────────
    df_pot = pd.DataFrame(all_pot).sort_values('time').reset_index(drop=True)
    df_thresh = pd.DataFrame(thresholds)
    
    df_pot.to_csv(os.path.join(output_dir, f'POT_NTR_TimeVarying_{target_events}.csv'), index=False)
    df_thresh.to_csv(os.path.join(output_dir, f'NTR_TimeVarying_Thresholds_{target_events}.csv'), index=False)
    
    
    #River Discharge
    POT_df = pd.read_csv(rf'{output_dir}/POT_NTR_TimeVarying_{target_events}.csv')
    POT_df['time'] = pd.to_datetime(POT_df['time'])
    POT_df = POT_df.set_index(['time'])
    
    df_RD = pd.read_csv(RD_File)
    df_RD['time'] = pd.to_datetime(df_RD[time_col_rd])
    df_RD = df_RD.drop(columns=[time_col_rd])
    df_RD = df_RD.set_index(['time'])
    
    # Ensure RD is daily (rounded to date)
    df_RD.index = df_RD.index.date  # Convert to datetime.date type
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in POT_df.index:
        event_date = event_time.date()  # extract the date part (YYYY-MM-DD)
    
        start_date = event_date - timedelta(days=2.5)
        end_date = event_date + timedelta(days=2.5)
    
        # Subset discharge data in that window
        window_data = df_RD.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    POT_df.index = pd.to_datetime(POT_df.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    POT_df = POT_df.copy()
    POT_df['date'] = POT_df.index.date  # datetime.date
    POT_df = POT_df.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'})  # 'date' is of type datetime.date
    
    # Step 3: Merge on 'date'
    merged_df = POT_df.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    merged_df = merged_df.drop(columns=['time'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['Time'] = pd.to_datetime(merged_df['Time'])
    
    merged_df.to_csv(rf'{output_dir}/POT_NTR_TimeVarying_RF_RD_{target_events}.csv', index=False)

C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\762217692.py:231: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\762217692.py:231: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')


## POT, NTR, Reanalysis, Stationary

In [ ]:
locations = ['OceanSprings']

# Name of "time" column in each dataset
time_col_ntr = 'time'
time_col_rf = 'time'
time_col_rd = 'valid_time'

# Name of variable column in each dataset
ntr_col = 'Average'
rd_col = 'basin_avg_dis24_m3s'
acc_hr = 'acc_24hr'

# Number of events to sample per year
target_events = 10  

core_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/POT_24hr_Jan26/Trivariate')
dataset_directory = Path(r'Synthetic_Scenarios_for_Compound_Coastal_Flooding/Datasets')

In [ ]:
for loc in locations:
    base_path = core_directory / Path(f'{loc}')
    
    NTR_File = rf'{dataset_directory}/Reanalysis/Compare_Points_{loc}_Avg.csv'
    RF_File = rf'{dataset_directory}/Reanalysis/APCP_Basin_Average_{loc}_Combined_accumulated.csv'
    RD_File = rf'{dataset_directory}/Reanalysis/GloFAS_{loc}_Basin_Avg.csv'
    
    df_ntr = pd.read_csv(NTR_File)
    df_ntr['time'] = pd.to_datetime(df_ntr[time_col_ntr])
    df_ntr = df_ntr.set_index('time')

    df_rf = pd.read_csv(RF_File)
    df_rf['time'] = pd.to_datetime(df_rf[time_col_rf])
    df_rf = df_rf.set_index('time')

    # ────────────────────────────────────────────────────────────────────────────────
    # Parameters
    # ────────────────────────────────────────────────────────────────────────────────
    ntr_series = df_ntr[ntr_col].dropna()
    decluster_window = timedelta(days=2.5)
    output_dir = rf'{base_path}/Reanalysis/Stationary'

    # ────────────────────────────────────────────────────────────────────────────────
    # Find constant threshold for 5 events/year
    # ────────────────────────────────────────────────────────────────────────────────
    n_years = ntr_series.index.year.nunique()
    total_target_events = target_events * n_years
    
    def count_events(threshold):
        selected = []
        for t, val in ntr_series.items():
            if val >= threshold:
                if any(abs((t - sel).days) < decluster_window.days for sel in selected):
                    continue
                selected.append(t)
        return len(selected)
    
    # Start from the initial threshold (top N values)
    sorted_values = ntr_series.sort_values(ascending=False)
    low, high = sorted_values.iloc[-1], sorted_values.iloc[0]  
    best_threshold = None
    
    # Binary search for the threshold that gives ~ target_events
    for _ in range(50):  # 50 iterations for convergence
        mid = (low + high) / 2
        events = count_events(mid)
        if events > total_target_events:
            low = mid
        else:
            high = mid
        best_threshold = mid
    
    threshold_value = best_threshold
    
    # Declustering window: ±2.5 days
    decluster_window = timedelta(days=2.5)
    
    # Step 1: Identify POT events (≥ threshold_value) with 2.5-day declustering
    pot_selected = []
    for t, val in ntr_series.sort_values(ascending=False).items():
        if val >= threshold_value:
            if any(abs((t - sel).days) <= decluster_window.days for sel in pot_selected):
                continue
            pot_selected.append(t)
    
    # Step 2: Mark all times within ±2.5 days of any POT event
    pot_exclusion_periods = [(event_time - decluster_window, event_time + decluster_window)
                             for event_time in pot_selected]
    
    def is_in_exclusion_period(time, exclusion_periods):
        return any(start <= time <= end for start, end in exclusion_periods)
    
    # Step 3: Filter below threshold and outside exclusion windows
    below_threshold_times = df_ntr[
        (df_ntr[ntr_col] < threshold_value) &
        (~df_ntr.index.map(lambda t: is_in_exclusion_period(t, pot_exclusion_periods)))
    ].index
    
    # Step 4: For each POT event, find max NTR in ±2.5 days
    pot_list = []
    for t in pot_selected:
        rf_window = df_rf.loc[t - decluster_window : t + decluster_window]
        max_rf= rf_window[acc_hr].max() if not rf_window.empty else np.nan
        pot_list.append({
            'time': t,
            'POT_NTR': df_ntr.loc[t, ntr_col],
            'Max_RF_in_5d': max_rf
        })
    
    df_pot_events = pd.DataFrame(pot_list).sort_values('time').reset_index(drop=True)
    
    output_below_file = os.path.join(output_dir, f'NTR_Stationary_Threshold_{round(threshold_value,2)}_{target_events}.csv')
    df_pot_events.to_csv(output_below_file, index=False)
    
    output_below_file = os.path.join(output_dir, f'POT_NTR_Stationary_{target_events}.csv')
    df_pot_events.to_csv(output_below_file, index=False)

    POT_df = pd.read_csv(rf'{output_dir}/POT_NTR_Stationary_{target_events}.csv')
    POT_df['time'] = pd.to_datetime(POT_df['time'])
    POT_df = POT_df.set_index(['time'])
    
    df_RD = pd.read_csv(RD_File)    
    df_RD['time'] = pd.to_datetime(df_RD[time_col_rd])
    df_RD = df_RD.drop(columns=[time_col_rd])
    df_RD = df_RD.set_index(['time'])
    
    # Ensure RD is daily (rounded to date)
    df_RD.index = df_RD.index.date 
    
    # Initialize result list
    all_max_series = []
    
    # Loop through each POT event time (as date)
    for event_time in POT_df.index:
        event_date = event_time.date() 
    
        start_date = event_date - timedelta(days=2.5)
        end_date = event_date + timedelta(days=2.5)
    
        # Subset discharge data in that window
        window_data = df_RD.loc[start_date:end_date]
    
        # Get max discharge values and store with event label
        max_rd_series = window_data.max()
        max_rd_series.name = event_date
        all_max_series.append(max_rd_series)
    
    # Combine all into a new DataFrame
    max_RD_df = pd.DataFrame(all_max_series)
    POT_df.index = pd.to_datetime(POT_df.index).round('H')
    
    # Step 1: Create a 'date' column from the POT_df index
    POT_df = POT_df.copy()
    POT_df['date'] = POT_df.index.date  
    POT_df = POT_df.reset_index()
    
    # Step 2: Reset index for max_RD_df and rename its index to 'date'
    max_RD_df = max_RD_df.copy()
    max_RD_df = max_RD_df.reset_index().rename(columns={'index': 'date'}) 
    
    # Step 3: Merge on 'date'
    merged_df = POT_df.merge(max_RD_df, on='date', how='left')
    merged_df = merged_df.drop(columns=['date'])
    merged_df = merged_df.drop(columns=['time'])
    merged_df = merged_df.dropna()
    
    merged_df = merged_df.rename(columns={rd_col:'Max_RD_in_5d'})
    merged_df['Time'] = pd.to_datetime(merged_df['Time'])
    
    merged_df.to_csv(rf'{output_dir}/POT_NTR_Stationary_RF_RD_{target_events}.csv', index=False)
    

C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\3044684352.py:212: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')
C:\Users\sadaf\AppData\Local\Temp\ipykernel_61520\3044684352.py:212: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  POT_df.index = pd.to_datetime(POT_df.index).round('H')
